Before running this code:
- Needs pandas, openpyxl and spacy, plus the spaCy model (python -m spacy download en_core_web_sm)
- Change Output_folder and Rater_folder in the import cell to the correct paths, e.g. r"C:\Users\USER\UNI OneDrive\PRIVATE FOLDER\Raw documents" (create the Rater files folder first)
- Change file_path in the cleaning cell to the real corpus file

PART ONE:
Run the top 4 code cells, in order. Then do the manual screening in Excel (screening.xlsx):
-- Websites tab: type each website's provider
-- Documents tab: put Yes or No in Include. If No, give the reason in Exclusion_Reason
-- Only fill in Provider on the Documents tab if it is different from the website's provider
-- For rows marked Check_Personal_Info, open cleaned_corpus.csv in VS Code (not Excel) and delete any personal information left in the Cleaned_Text column
Save and close both files.
Do NOT re-run the top 4 cells after this point: it would overwrite cleaned_corpus.csv.

PART TWO:
If you restarted VS Code, run the import cell again, then skip to the load cell and run from there downwards.
-- If the load cell stops with missing providers, fill them in, save, and re-run the load cell
-- Open provider_key.csv and check no organisation appears twice under different spellings (e.g. a full name and an abbreviation). If one does, make the names match in screening.xlsx and re-run from the load cell
-- Check passages_removed_by_keyword_filter.csv. If relevant passages were removed, add the word to ADHD_words and re-run from the passage table cell
-- If there are fewer than 500 passages, collect more documents before going any further. If you re-scrape, move the old screening.xlsx into a separate folder and start Part One again
-- Check personal_info_check.csv. Remove any real leftovers from the Cleaned_Text column of cleaned_corpus.csv (in VS Code, not Excel), then re-run from the load cell

PART THREE (point of no return):
Run the split. It can only be run once; do not change the seed.
Check the rater sheets contain only Passage_ID, Passage_Text and the empty coding columns.
Check audit_summary.csv has the numbers needed for the Method.


In [ ]:
# Import libraries
import pandas as pd
import re
import spacy as sp
import random
import os
import urllib.parse
from openpyxl.styles import Alignment
from openpyxl.worksheet.datavalidation import DataValidation
# Loading the English language model from spaCy
nlp = sp.load("en_core_web_sm")
Output_folder = ''
Rater_folder = ''
random_seed = 2026

In [ ]:
# Cleaning the raw data
file_path = 'dummy_corpus.csv'  # Path to the CSV file containing the text data
corpus = pd.read_csv(file_path)
corpus['Raw_Text'] = corpus['Raw_Text'].fillna('').astype(str) # Make every entry a string
corpus['Raw_Text'] = [re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', text) for text in corpus['Raw_Text']]  # Remove invisible PDF characters that Excel can't save
corpus = corpus[[len(text.split()) > 1 for text in corpus['Raw_Text']]]  # Filter out rows with only 1 word

In [ ]:
# Removing identifying information
Email = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
Phone = re.compile(r'(?:\+44\s?(?:\(0\)\s?)?|\(?\b0)\d{2,4}\)?[\s-]?\d{3,4}[\s-]?\d{3,4}\b')   # also catches (01206) and +44 (0)
Titled_name = re.compile(r'\b(?:Dr|Mr|Mrs|Ms|Prof|Professor|Miss)\.?\s+[A-Z][a-z]+(?:[\s-][A-Z][a-z]+)*')

def remove_identifying_info(text):
    counts = {}
    text, counts['emails'] = Email.subn('[EMAIL]', text)
    text, counts['phones'] = Phone.subn('[PHONE]', text)
    text, counts['names'] = Titled_name.subn('[NAME]', text)

    # Non-titled names found by spaCy
    people = [ent.text for ent in nlp(text).ents if ent.label_ == 'PERSON']
    for name in people:
        text = re.sub(rf'\b{re.escape(name)}\b', '[NAME]', text)   # whole words only
    counts['other_names'] = len(people)
    return text, counts

results = [remove_identifying_info(text) for text in corpus['Raw_Text']]
corpus['Cleaned_Text'] = [result[0] for result in results]
corpus = corpus.join(pd.DataFrame([result[1] for result in results], index=corpus.index))

In [ ]:
# Giving each doc a fixed ID and saving the files
corpus['Doc_ID'] = [f"D{i:03d}" for i in range(1, len(corpus) + 1)]
corpus['Website'] = [urllib.parse.urlparse(url).netloc.removeprefix("www.") for url in corpus['URL']]
corpus.to_csv(os.path.join(Output_folder, "cleaned_corpus.csv"), index=False, encoding="utf-8")

screening_path = os.path.join(Output_folder, "screening.xlsx")
if os.path.exists(screening_path):
    print("screening.xlsx already exists, so it was not overwritten (your decisions are safe)")
else:
    # Documents tab: one row per document
    documents = corpus[['Doc_ID', 'Website', 'URL', 'Document_Type']].copy()
    documents['Preview'] = [text[:300] for text in corpus['Cleaned_Text']]
    personal_info_total = corpus[['emails', 'phones', 'names', 'other_names']].sum(axis=1)
    documents['Check_Personal_Info'] = ['Yes' if total > 0 else '' for total in personal_info_total]
    documents['Include'] = ''              # fill in: Yes or No
    documents['Exclusion_Reason'] = ''     # fill in if No, e.g. paediatric, off-topic
    documents['Provider'] = ''             # only fill in if different from the website's provider

    # Websites tab: name each website's provider once
    websites = documents.groupby('Website').size().reset_index(name='Documents')
    websites['Provider'] = ''              # fill in: the organisation's name

    with pd.ExcelWriter(screening_path) as writer:
        documents.to_excel(writer, sheet_name='Documents', index=False)
        websites.to_excel(writer, sheet_name='Websites', index=False)
    print(f"Saved screening.xlsx: {len(documents)} documents from {len(websites)} websites")

The following sections to be run AFTER manually checking corpus and adding providers

In [ ]:
# Load the current files
corpus = pd.read_csv(os.path.join(Output_folder, "cleaned_corpus.csv"))
screening_path = os.path.join(Output_folder, "screening.xlsx")
documents = pd.read_excel(screening_path, sheet_name='Documents', dtype=str).fillna('')
websites = pd.read_excel(screening_path, sheet_name='Websites', dtype=str).fillna('')

# Each document's provider: its own entry if filled in, otherwise its website's provider
provider_by_website = dict(zip(websites['Website'], websites['Provider']))

def choose_provider(document_provider, website):
    if document_provider.strip():
        return document_provider.strip()
    return provider_by_website.get(website, '').strip()

documents['Provider'] = [choose_provider(provider, website)
                         for provider, website in zip(documents['Provider'], documents['Website'])]

# Every document needs Yes or No, so a skipped row can't be excluded by accident
answers = documents['Include'].str.strip().str.lower()
not_screened = documents[[answer not in ['yes', 'no'] for answer in answers]]
if len(not_screened) > 0:
    raise ValueError(f"Include must be Yes or No for every document. Check: {', '.join(not_screened['Doc_ID'])}")

included = documents[answers == 'yes'].copy()
excluded = documents[answers == 'no']

missing_provider = included[included['Provider'] == '']
if len(missing_provider) > 0:
    raise ValueError(f"Included documents with no provider: {', '.join(missing_provider['Doc_ID'])}")

# Swapping provider names for codes (P01, P02...); the key stays in the restricted folder
provider_names = sorted(included['Provider'].unique())
provider_codes = {name: f"P{number:02d}" for number, name in enumerate(provider_names, 1)}
pd.DataFrame(provider_codes.items(), columns=['Provider', 'Provider_ID']).to_csv(
    os.path.join(Output_folder, "provider_key.csv"), index=False, encoding="utf-8")
included['Provider_ID'] = included['Provider'].map(provider_codes)

# Matching on Doc_ID and URL, so screening decisions can't attach to the wrong document
corpus = corpus.merge(included[['Doc_ID', 'URL', 'Provider_ID']], on=['Doc_ID', 'URL'])
if len(corpus) != len(included):
    raise ValueError("screening.xlsx does not match cleaned_corpus.csv (some Doc_IDs point to different URLs)")
print(f"Documents screened: {len(documents)} | included: {len(corpus)} | providers: {len(provider_names)}")
print("Exclusion reasons:", excluded['Exclusion_Reason'].value_counts().to_dict())


In [ ]:
# Defining boilerplate phrases to filter out and setting min and max words
boilerplate_phrases = ['click here', 'read more', 'subscribe', 'advertisement', 'sponsored',
                       'skip to main', 'skip to content', 'privacy policy', 'terms of service',
                       'cookie', 'follow us', 'back to top', 'print this page']
Max_words = 200
Min_words = 20

In [ ]:
# Defining headers and non-substantive phrases to filter out
def is_heading(block):
    return len(block.split()) <= 12 and not block.rstrip().endswith(('.', '!', '?', ':'))

def is_non_substantive(block):
    words = len(re.findall(r'\w+', block)) 
    lower = block.lower()
    if words < 25 and any(phrase in lower for phrase in boilerplate_phrases):
        return True
    if words < 25 and block.count('|') + block.count('>') >= 2:
        return True
    if words < 40 and ('[EMAIL]' in block or '[PHONE]' in block):
        return True
    if words > 30 and not any(p in block for p in '.!?'):
        return True
    return False

In [ ]:
# Splitting the documents into passages
def make_passages(text):
    # 1. Paragraph and section blocks (the scraper puts a blank line between them)
    blocks = []
    for raw in re.split(r'\n\s*\n', text):
        lines = [line.strip() for line in raw.split('\n') if line.strip()]
        if not lines:
            continue
        if len(lines) > 1 and is_heading(lines[0]):
            blocks += [lines[0], ' '.join(lines[1:])]
        else:
            blocks.append(' '.join(lines))
    blocks = [b for b in blocks if not is_non_substantive(b)]
    
    # 2. Group into sections: a heading starts a new section, the text below it joins that section.
    #    Short lines followed by more short lines are bullet points, so they stay in the current section
    sections = []
    for i, block in enumerate(blocks):
        next_block = blocks[i + 1] if i + 1 < len(blocks) else ''
        starts_section = is_heading(block) and not is_heading(next_block)
        if starts_section or not sections:
            sections.append(block)
        else:
            sections[-1] += '\n' + block

    # 3. Merge a very short section (e.g. a bullet point or a heading on its own) into the next one, if the result stays within the limit
    merged = []
    for section in sections:
        if merged and len(merged[-1].split()) < Min_words \
                and len(merged[-1].split()) + len(section.split()) <= Max_words:
            merged[-1] += '\n' + section
        else:
            merged.append(section)

    # 4. Split long sections at the nearest line break or sentence end below the limit
    passages = []
    for section in merged:
        if len(section.split()) <= Max_words:
            passages.append(section)
            continue
        pieces = []
        for line in section.split('\n'):                  # bullet points are separate lines
            for sentence in nlp(line).sents:
                words = sentence.text.split()
                # a single sentence over the limit (rare, e.g. no full stops) is cut every Max_words words
                for start in range(0, len(words), Max_words):
                    pieces.append(' '.join(words[start:start + Max_words]))
        current, count = [], 0
        for piece in pieces:
            n = len(piece.split())
            if current and count + n > Max_words:
                passages.append(' '.join(current))
                current, count = [], 0
            current.append(piece)
            count += n
        if current:
            passages.append(' '.join(current))
    return passages


In [ ]:
# Building the passage table with one row per passage, still linked to its document and provider
passages = []
for row in corpus.itertuples():
    for n, passage in enumerate(make_passages(row.Cleaned_Text), 1):
        passages.append({
            "Passage_ID": f"{row.Doc_ID}-{n:02d}",   # document ID - passage number
            "Doc_ID": row.Doc_ID,
            "Provider_ID": row.Provider_ID,
            "Passage_Text": passage,
            "Word_Count": len(passage.split()),
        })
passages = pd.DataFrame(passages)

In [ ]:
# Keeping only passages that mention broad ADHD related words, one of the four target difficulties, or a form of support
ADHD_acronyms = ['ADHD', 'ADD']        # capitals only, so the word "add" doesn't count
ADHD_words = ['attention deficit', 'attention-deficit', 'hyperactiv', 'inattent',
              'concentrat', 'distract', 'focus', 'memory', 'forget',
              'emotion', 'mood', 'irritab', 'frustrat', 'motivat', 'finishing tasks', 'finish tasks',
              'support', 'therap', 'medicat', 'intervention', 'coaching', 'psychoeducation', 'cbt']

def mentions_adhd(text):
    lower = text.lower()
    has_acronym = any(re.search(rf'\b{acronym}\b', text) for acronym in ADHD_acronyms)
    has_word = any(word in lower for word in ADHD_words)
    return has_acronym or has_word

passages_before_filter = len(passages)
keep = [mentions_adhd(text) for text in passages['Passage_Text']]
removed = passages[[not k for k in keep]]
removed.to_csv(os.path.join(Output_folder, "passages_removed_by_keyword_filter.csv"), index=False, encoding="utf-8")
passages = passages[keep].reset_index(drop=True)
print(f"Keyword filter: kept {len(passages)} of {passages_before_filter} passages "
      f"(the {len(removed)} removed are saved to passages_removed_by_keyword_filter.csv for checking)")

In [ ]:
# Printing corpus description
print("Documents:", len(corpus))
print("Passages:", len(passages))
print("Words per passage: median", passages['Word_Count'].median(), "| max", passages['Word_Count'].max())


In [ ]:
# Manual check for identifying information
needs_check = passages[passages['Passage_Text'].str.contains(r'\[(?:NAME|EMAIL|PHONE)\]')]
needs_check.to_csv(os.path.join(Output_folder, "personal_info_check.csv"), index=False, encoding="utf-8")
print(f"Passages to check by hand for personal information: {len(needs_check)}")

After this point, the passages are segmented and split into the different sections (development, calibration and the held out test set)

In [ ]:
# Splitting the passages into the sections
Split_shares = {"development": 0.58, "calibration": 0.17, "test": 0.25}

def split_by_provider(passages, provider_column="Provider_ID", ratio=Split_shares, seed=random_seed):
    providers = sorted(passages[provider_column].unique())
    if len(providers) < len(ratio):
        raise ValueError(f"Need at least {len(ratio)} providers to make {len(ratio)} sets")
    random.Random(seed).shuffle(providers)

    set_names = list(ratio)
    fill_up_to, running_share = [], 0
    for set_name in set_names:
        running_share += ratio[set_name]
        fill_up_to.append(running_share * (len(passages)))

    passages_per_provider = passages[provider_column].value_counts()
    provider_to_set, passages_assigned, current_set_num = {}, 0, 0
    for provider in providers:
        if current_set_num < len(set_names) - 1 and passages_assigned >= fill_up_to[current_set_num]:
            current_set_num += 1
        provider_to_set[provider] = set_names[current_set_num]
        passages_assigned += passages_per_provider[provider]

    split_passages = passages.copy()
    split_passages["Split"] = split_passages[provider_column].map(provider_to_set)
    return split_passages

In [ ]:
# Checking for data leakage
def check_for_leakage(split_passages, provider_column="Provider_ID", text_column="Passage_Text"):
    sets_per_provider = split_passages.groupby(provider_column)["Split"].nunique()
    if (sets_per_provider > 1).any():
        raise ValueError("Leakage: a provider appears in more than one set")

    normalised = split_passages[text_column].str.lower().str.split().str.join(" ")
    sets_per_text = split_passages.groupby(normalised)["Split"].nunique()
    shared_texts = sets_per_text[sets_per_text > 1].index
    print(f"Identical passages found in more than one set: {len(shared_texts)}")
    return normalised.isin(shared_texts)

In [ ]:
# Running the split (only ever once: the split is frozen as soon as it is saved)
split_path = os.path.join(Output_folder, "passage_split.csv")
if os.path.exists(split_path):
    raise FileExistsError("passage_split.csv already exists. The split is frozen; "
                          "only delete it if no coding has started.")

split_passages = split_by_provider(passages)
if split_passages['Split'].nunique() < len(Split_shares):
    raise ValueError("A set ended up empty; there are too few providers for a three-way split")

# Identical passages in more than one set are kept in development only
is_duplicate = check_for_leakage(split_passages)
keep = [not (duplicate and set_name != 'development')
        for duplicate, set_name in zip(is_duplicate, split_passages['Split'])]
split_passages = split_passages[keep]

print("Passages per set:", split_passages['Split'].value_counts().to_dict())
print("Share of passages per set:", split_passages['Split'].value_counts(normalize=True).round(3).to_dict())
split_passages.to_csv(split_path, index=False, encoding="utf-8")

In [ ]:
# Coding sheets for the rater: pseudonymised passages only, in a random order
# The choices in each drop-down list (from the coding framework)
coding_options = {
    'Primary_Label': ['Recognition only (0)', 'Provision (1)', 'Not relevant'],
    'Provision_Level': ['No support (0)', 'Signposting (1)', 'General support (2)', 'Targeted support (3)'],
    'ATT': ['Yes'], 'PER': ['Yes'], 'ESR': ['Yes'], 'WM': ['Yes'], 'Other': ['Yes'],   # one column per domain, as a passage can have several
    'Uncertain': ['Yes'],
}
coding_columns = list(coding_options) + ['Notes']

def make_coding_sheet(chosen_passages, file_name):
    sheet = chosen_passages[['Passage_ID', 'Passage_Text']].copy()
    for column in coding_columns:
        sheet[column] = ''                          # empty columns for the rater to fill in

    with pd.ExcelWriter(os.path.join(Rater_folder, file_name), engine='openpyxl') as writer:
        sheet.to_excel(writer, sheet_name='Coding', index=False)
        ws = writer.sheets['Coding']
        ws.freeze_panes = 'C2'                      # headings and passage text stay in view when scrolling
        last_row = len(sheet) + 1
        for number, column in enumerate(sheet.columns, 1):
            letter = ws.cell(row=1, column=number).column_letter
            ws.column_dimensions[letter].width = 22
            if column in coding_options:            # drop-down list, so only valid codes can be entered
                dropdown = DataValidation(type='list', formula1='"' + ','.join(coding_options[column]) + '"')
                ws.add_data_validation(dropdown)
                dropdown.add(f'{letter}2:{letter}{last_row}')
        ws.column_dimensions['A'].width = 12
        ws.column_dimensions['B'].width = 90
        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical='top')

development = split_passages[split_passages['Split'] == 'development']
test = split_passages[split_passages['Split'] == 'test']

rater_stage1 = development.sample(frac=1/3, random_state=random_seed)   # a random third (same third every run)
rater_stage2 = test.sample(frac=1, random_state=random_seed)            # all of the test set, shuffled

make_coding_sheet(rater_stage1, "rater_stage1_development.xlsx")
make_coding_sheet(rater_stage2, "rater_stage2_test.xlsx")
print(f"Rater coding sheets saved: Stage 1 = {len(rater_stage1)} passages, Stage 2 = {len(rater_stage2)} passages")

In [ ]:
# Audit summary for the Method (counts only, nothing identifying)
set_sizes = split_passages['Split'].value_counts()
audit = {
    "Documents screened": len(documents),
    "Documents included": len(corpus),
    "Providers": len(provider_names),
    "Emails removed": int(corpus['emails'].sum()),
    "Phone numbers removed": int(corpus['phones'].sum()),
    "Titled names removed": int(corpus['names'].sum()),
    "Other names removed": int(corpus['other_names'].sum()),
    "Passages before keyword filter": passages_before_filter,
    "Passages after keyword filter": len(passages),
    "Passages after duplicate removal": len(split_passages),
    "Development passages": int(set_sizes.get('development', 0)),
    "Calibration passages": int(set_sizes.get('calibration', 0)),
    "Test passages": int(set_sizes.get('test', 0)),
    "Rater Stage 1 passages": len(rater_stage1),
    "Rater Stage 2 passages": len(rater_stage2),
}
audit = pd.Series(audit, name="Count")
audit.to_csv(os.path.join(Output_folder, "audit_summary.csv"), encoding="utf-8")
print(audit.to_string())